### Purpose

```sql
This notebook is designed to do following tasks:
- load train_transaction and train_identity
- join on TransactionID
- sort by TransactionDT
- create one train/validation split
- sanity check row counts, fraud rate, identity coverage
```

In [1]:
import pandas as pd

##### 1. Load train_transaction and train_identity

In [2]:
df_train_txn=pd.read_csv('../data/raw/train_transaction.csv')

In [3]:
df_train_identity=pd.read_csv('../data/raw/train_identity.csv')

In [4]:
df_train_txn.shape, df_train_identity.shape

((590540, 394), (144233, 41))

In [5]:
display(df_train_txn.head(2)), display(df_train_identity.head(2))

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device


(None, None)

#### 2. join on TransactionID

In [6]:
df_train_txn["TransactionID"].nunique(), df_train_identity["TransactionID"].nunique()

(590540, 144233)

In [7]:
df_joined=pd.merge(df_train_txn,df_train_identity,how="outer",on=["TransactionID"])

In [8]:
df_joined.shape

(590540, 434)

In [10]:
#Missing rate by Columns - Nulls
missing_summary=(
    df_joined.isna().mean().sort_values(ascending=False).rename("missing_rate").reset_index().rename(columns={"index":"column"})
)

display(missing_summary)

,column,missing_rate
0,id_24,0.991962
1,id_25,0.991310
2,id_07,0.991271
3,id_08,0.991271
4,id_21,0.991264
...,...,...
429,C11,0.000000
430,C14,0.000000
431,C13,0.000000
432,C12,0.000000


#### 3. Sort by TransactionDT

In [11]:
#Discard old index and build a new index
df_joined=df_joined.sort_values(by="TransactionDT",ascending=True).reset_index(drop=True)

#### 4. create one train/validation split

In [16]:
#80-20 split based on timestamp

def time_split(df: pd.DataFrame, fraction: float=0.2):
    split_index=int(len(df)*(1-fraction)) #get train end index
    
    #train will start from zero and go till split index
    train_df=df.iloc[:split_index].copy()
    valid_df=df.iloc[split_index:].copy()
    #Copy is created so that any changes in df will not impact change in train_df or valid_df

    return train_df, valid_df

In [21]:
train_df_1, valid_df_1=time_split(df_joined,0.2)

#### 5. Sanity Checks

In [22]:
#Counts
print(len(df_joined))
print(train_df_1.shape)
print(valid_df_1.shape)

590540
(472432, 434)
(118108, 434)


In [24]:
#Timestamp should not overlap
print(train_df_1["TransactionDT"].min(),train_df_1["TransactionDT"].max())
print(valid_df_1["TransactionDT"].min(),valid_df_1["TransactionDT"].max())

86400 12192842
12192900 15811131


In [27]:
#Fraud Rate
print(f"train fraud rate : {train_df_1["isFraud"].mean():.2%}")
print(f"valid fraud rate : {valid_df_1["isFraud"].mean():.2%}")

train fraud rate : 3.51%
valid fraud rate : 3.44%


```sql

Outcomes:-

At this point we have:
- one joined training dataset
- rows sorted by `TransactionDT`
- one out-of-time train / validation split
- basic sanity checks on fraud rate and identity coverage

Next step:
- build the first baseline model on top of this split
```